# Upload one existing Kaggle keyframe shard directly to Cloudflare R2

Use a **CPU** notebook and enable **Internet**. Attach exactly the Kaggle dataset for the shard selected below. Add these Kaggle User Secrets and grant this notebook access to them: `R2_ACCESS_KEY_ID`, `R2_SECRET_ACCESS_KEY`, `R2_S3_CLIENT_URL`, and `R2_BUCKET`.

The notebook does not download or extract the dataset into `/kaggle/working`. It reads the mounted PNG files in `/kaggle/input` and uploads them directly as `fred/data/extracted_keyframes/<VIDEO_ID>/<FRAME>.png`. It validates the shard before uploading and writes only that shard's manifest, so distinct shards may run in parallel.


In [ ]:
# Change only this value for each notebook copy.
SHARD_ID = "L30_p01"

R2_PREFIX = "fred"
MAX_WORKERS = 16       # 8-16 is conservative; reduce if R2 returns throttling errors
PROGRESS_EVERY = 100

SOURCES = {
    "L21_p01": ("trungdangtapcode/aic-keyframes-l21-p01-output", 7737, 14),
    "L21_p02": ("trungdangtapcode/aic-keyframes-l21-p02-output", 8241, 15),
    "L22_p01": ("trungdangtapcode/aic-keyframes-l22-p01-output", 8852, 15),
    "L22_p02": ("trungdangtapcode/aic-keyframes-l22-p02-output", 9518, 16),
    "L23_p01": ("trungdangtapcode/aic-keyframes-l23-p01-output", 4872, 25),
    "L24_p01": ("trungdangtapcode/aic-keyframes-l24-p01-output", 5525, 20),
    "L24_p02": ("trungdangtapcode/aic-keyframes-l24-p02-output", 5565, 23),
    "L25_p01": ("trungdangtapcode/aic-keyframes-l25-p01-output", 9134, 13),
    "L25_p02": ("trungdangtapcode/aic-keyframes-l25-p02-output", 9229, 13),
    "L25_p03": ("trungdangtapcode/aic-keyframes-l25-p03-output", 9787, 14),
    "L25_p04": ("trungdangtapcode/aic-keyframes-l25-p04-output", 8791, 11),
    "L25_p05": ("trungdangtapcode/aic-keyframes-l25-p05-output", 9458, 13),
    "L25_p06": ("trungdangtapcode/aic-keyframes-l25-p06-output", 9723, 13),
    "L25_p07": ("trungdangtapcode/aic-keyframes-l25-p07-output", 9116, 11),
    "L26_p01": ("trungdangtapcode/aic-keyframes-l26-p01-output", 9869, 62),
    "L26_p02": ("trungdangtapcode/aic-keyframes-l26-p02-output", 9894, 61),
    "L26_p03": ("trungdangtapcode/aic-keyframes-l26-p03-output", 9911, 61),
    "L26_p04": ("trungdangtapcode/aic-keyframes-l26-p04-output", 9837, 62),
    "L26_p05": ("trungdangtapcode/aic-keyframes-l26-p05-output", 9930, 62),
    "L26_p06": ("trungdangtapcode/aic-keyframes-l26-p06-output", 9898, 64),
    "L26_p07": ("trungdangtapcode/aic-keyframes-l26-p07-output", 9809, 63),
    "L26_p08": ("trungdangtapcode/aic-keyframes-l26-p08-output", 9901, 63),
    "L27_p01": ("trungdangtapcode/aic-keyframes-l27-p01-output", 4723, 16),
    "L28_p01": ("trungdangtapcode/aic-keyframes-l28-p01-output", 6878, 12),
    "L28_p02": ("trungdangtapcode/aic-keyframes-l28-p02-output", 6781, 12),
    "L29_p01": ("trungdangtapcode/aic-keyframes-l29-p01-output", 6405, 12),
    "L29_p02": ("trungdangtapcode/aic-keyframes-l29-p02-output", 5921, 11),
    "L30_p01": ("trungdangtapcode/aic-keyframes-l30-p01-output", 10283, 96),
}

assert SHARD_ID in SOURCES, f"Unknown SHARD_ID: {SHARD_ID}"
assert 1 <= MAX_WORKERS <= 32
SOURCE_DATASET, EXPECTED_FRAMES, EXPECTED_VIDEOS = SOURCES[SHARD_ID]
print(f"Configured {SHARD_ID}: {SOURCE_DATASET}, {EXPECTED_FRAMES:,} frames, {EXPECTED_VIDEOS} videos")


In [ ]:
import concurrent.futures
import json
import re
import subprocess
import sys
import threading
import time
import uuid
import zlib
from datetime import datetime, timezone
from pathlib import Path

try:
    import boto3
    from botocore.config import Config
    from botocore.exceptions import ClientError
except ModuleNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "boto3"], check=True)
    import boto3
    from botocore.config import Config
    from botocore.exceptions import ClientError

from kaggle_secrets import UserSecretsClient

secret_client = UserSecretsClient()
R2_ACCESS_KEY_ID = secret_client.get_secret("R2_ACCESS_KEY_ID").strip()
R2_SECRET_ACCESS_KEY = secret_client.get_secret("R2_SECRET_ACCESS_KEY").strip()
R2_S3_CLIENT_URL = secret_client.get_secret("R2_S3_CLIENT_URL").strip().rstrip("/")
R2_BUCKET = secret_client.get_secret("R2_BUCKET").strip()
R2_PREFIX = R2_PREFIX.strip().strip("/")
assert all((R2_ACCESS_KEY_ID, R2_SECRET_ACCESS_KEY, R2_S3_CLIENT_URL, R2_BUCKET, R2_PREFIX))
assert R2_S3_CLIENT_URL.startswith("https://"), "R2_S3_CLIENT_URL must be an HTTPS S3 endpoint"

client_local = threading.local()

def r2_client():
    client = getattr(client_local, "client", None)
    if client is None:
        client = boto3.client(
            "s3",
            endpoint_url=R2_S3_CLIENT_URL,
            region_name="auto",
            aws_access_key_id=R2_ACCESS_KEY_ID,
            aws_secret_access_key=R2_SECRET_ACCESS_KEY,
            config=Config(
                signature_version="s3v4",
                retries={"max_attempts": 10, "mode": "standard"},
                connect_timeout=30,
                read_timeout=300,
                max_pool_connections=max(16, MAX_WORKERS * 2),
                s3={"addressing_style": "path"},
            ),
        )
        client_local.client = client
    return client

def object_head(key):
    try:
        return r2_client().head_object(Bucket=R2_BUCKET, Key=key)
    except ClientError as exc:
        response = exc.response
        status = response.get("ResponseMetadata", {}).get("HTTPStatusCode")
        code = str(response.get("Error", {}).get("Code", ""))
        if status == 404 or code in {"404", "NoSuchKey", "NotFound"}:
            return None
        raise

# Verify write, read, and delete access without requiring ListBucket.
probe_key = f"{R2_PREFIX}/manifests/.upload-probes/kaggle-{uuid.uuid4().hex}"
probe_payload = b"kaggle-r2-upload-probe\n"
try:
    r2_client().put_object(Bucket=R2_BUCKET, Key=probe_key, Body=probe_payload, ContentType="text/plain")
    probe = r2_client().get_object(Bucket=R2_BUCKET, Key=probe_key)
    try:
        assert probe["Body"].read() == probe_payload
    finally:
        probe["Body"].close()
finally:
    r2_client().delete_object(Bucket=R2_BUCKET, Key=probe_key)
assert object_head(probe_key) is None, "R2 probe cleanup failed"
print(f"R2 write/read/delete probe passed; bucket={R2_BUCKET!r}, prefix={R2_PREFIX!r}")


In [ ]:
# Locate and fully validate the attached shard before changing R2.
INPUT_ROOT = Path("/kaggle/input")
dataset_owner, dataset_slug = SOURCE_DATASET.split("/", 1)
candidate_roots = [
    INPUT_ROOT / dataset_slug,
    INPUT_ROOT / "datasets" / dataset_owner / dataset_slug,
]
dataset_root = next((path for path in candidate_roots if path.is_dir()), None)
assert dataset_root is not None, (
    f"Attach Kaggle dataset {SOURCE_DATASET!r}. Expected one of {candidate_roots}; "
    f"found {[p.name for p in INPUT_ROOT.iterdir()] if INPUT_ROOT.is_dir() else 'no /kaggle/input'}"
)

frame_re = re.compile(r"^(L\d+_V\d+)/(\d{5})\.png$", re.IGNORECASE)
expected_collection = SHARD_ID.split("_", 1)[0].upper()
entries = []
destinations = set()
video_ids = set()

for path in dataset_root.rglob("*"):
    if not path.is_file() or path.suffix.lower() != ".png":
        continue
    parts = path.parts
    assert "extracted_keyframes" in parts, f"PNG outside extracted_keyframes: {path}"
    marker = parts.index("extracted_keyframes")
    relative = "/".join(parts[marker + 1:])
    match = frame_re.fullmatch(relative)
    assert match, f"Invalid keyframe path: {path}"
    video_id = match.group(1).upper()
    assert video_id.startswith(expected_collection + "_V"), f"Wrong collection in {path}"
    assert relative not in destinations, f"Duplicate destination: {relative}"
    with path.open("rb") as stream:
        assert stream.read(8) == b"\x89PNG\r\n\x1a\n", f"Bad PNG signature: {path}"
    size = path.stat().st_size
    assert size > 8, f"Empty/truncated PNG: {path}"
    destinations.add(relative)
    video_ids.add(video_id)
    entries.append((relative, path, size))

entries.sort(key=lambda item: item[0])
assert len(entries) == EXPECTED_FRAMES, f"Frame count mismatch: {len(entries):,} != {EXPECTED_FRAMES:,}"
assert len(video_ids) == EXPECTED_VIDEOS, f"Video count mismatch: {len(video_ids)} != {EXPECTED_VIDEOS}"
assert entries, "No PNG files found"
print(f"VALIDATED {SHARD_ID}: {len(entries):,} real PNG files across {len(video_ids)} videos")
print("First destination:", f"{R2_PREFIX}/data/extracted_keyframes/{entries[0][0]}")
print("Last destination: ", f"{R2_PREFIX}/data/extracted_keyframes/{entries[-1][0]}")


In [ ]:
SOURCE_URL = f"https://www.kaggle.com/datasets/{SOURCE_DATASET}"

def upload_one(entry):
    relative, path, expected_size = entry
    key = f"{R2_PREFIX}/data/extracted_keyframes/{relative}"
    payload = path.read_bytes()
    assert len(payload) == expected_size, f"Source changed while reading: {path}"
    assert payload[:8] == b"\x89PNG\r\n\x1a\n", f"Bad PNG signature: {path}"
    crc32 = f"{zlib.crc32(payload) & 0xffffffff:08x}"

    head = object_head(key)
    if head is not None and int(head.get("ContentLength", -1)) == expected_size:
        result = "skipped"
    else:
        response = r2_client().put_object(
            Bucket=R2_BUCKET,
            Key=key,
            Body=payload,
            ContentLength=expected_size,
            ContentType="image/png",
            Metadata={"source-dataset": SOURCE_DATASET, "source-crc32": crc32},
        )
        status = response.get("ResponseMetadata", {}).get("HTTPStatusCode")
        assert status in {200, 201}, f"R2 PUT returned HTTP {status} for {key}"
        verified = object_head(key)
        assert verified is not None and int(verified.get("ContentLength", -1)) == expected_size, (
            f"R2 size verification failed for {key}"
        )
        result = "uploaded"

    return result, {
        "key": key,
        "size": expected_size,
        "source": SOURCE_URL,
        "source_member": path.relative_to(dataset_root).as_posix(),
        "crc32": crc32,
    }

started = time.monotonic()
records = []
uploaded = skipped = 0
with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(upload_one, entry) for entry in entries]
    for completed, future in enumerate(concurrent.futures.as_completed(futures), 1):
        result, record = future.result()
        records.append(record)
        uploaded += result == "uploaded"
        skipped += result == "skipped"
        if completed % PROGRESS_EVERY == 0 or completed == len(futures):
            elapsed = max(time.monotonic() - started, 0.001)
            print(
                f"{SHARD_ID}: {completed:,}/{len(futures):,}; uploaded={uploaded:,}; "
                f"skipped={skipped:,}; rate={completed / elapsed:.1f} objects/s",
                flush=True,
            )

records.sort(key=lambda record: record["key"])
assert len(records) == EXPECTED_FRAMES


In [ ]:
# Publish this shard manifest only after every object passed size verification.
manifest_payload = b"".join(
    (json.dumps(record, sort_keys=True, separators=(",", ":")) + "\n").encode("utf-8")
    for record in records
)
manifest_key = f"{R2_PREFIX}/manifests/keyframes/{SHARD_ID}.jsonl"
r2_client().put_object(
    Bucket=R2_BUCKET,
    Key=manifest_key,
    Body=manifest_payload,
    ContentLength=len(manifest_payload),
    ContentType="application/x-ndjson",
    Metadata={"record-count": str(len(records))},
)
manifest_head = object_head(manifest_key)
assert manifest_head is not None and int(manifest_head["ContentLength"]) == len(manifest_payload)

summary = {
    "shard_id": SHARD_ID,
    "dataset": SOURCE_DATASET,
    "completed_at": datetime.now(timezone.utc).isoformat(),
    "manifest_key": manifest_key,
    "object_count": len(records),
    "uploaded": uploaded,
    "skipped": skipped,
    "total_bytes": sum(record["size"] for record in records),
    "video_count": len(video_ids),
}
summary_path = Path("/kaggle/working") / f"{SHARD_ID}.r2-summary.json"
summary_path.write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print(json.dumps(summary, indent=2))
print(f"DONE {SHARD_ID}: all {len(records):,} objects verified; manifest={manifest_key}")
